# Resume PPO V9 validation on Google Colab

This notebook resumes only missing `v9spec1` validation cells. It verifies commit `2bbad36`, victimagent `f4988db`, canonical data hashes, and append-only attempts before training. It does **not** run final training or access the 2022–2025 known period.

Before running, upload `artifacts/ppo_v9_code_2bbad36.bundle` and `artifacts/ppo_v9_state_after20.zip` to `MyDrive/victim_ppo_v9/`. Put the six raw CSV files in `MyDrive/victim_ppo_v9/raw/`. Never upload licensed raw data to GitHub.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
DRIVE = Path('/content/drive/MyDrive/victim_ppo_v9')
RAW = DRIVE / 'raw'
STATE = DRIVE / 'state'
REPO_URL = 'https://github.com/RL-trading-agent-robustness-evaluation/ppo_agent.git'
CODE_BUNDLE = DRIVE / 'ppo_v9_code_2bbad36.bundle'
STATE_ARCHIVE = DRIVE / 'ppo_v9_state_after20.zip'
REPO_COMMIT = '2bbad36'
VICTIM_COMMIT = 'f4988db5c98d9248533c4ca852cc48ca3f6b7aad'


In [ ]:
import os, shutil, subprocess
def run(*args, cwd=None):
    print('+', *map(str, args), flush=True)
    subprocess.run([str(x) for x in args], cwd=cwd, check=True)
repo = Path('/content/victim_ppo')
if repo.exists(): shutil.rmtree(repo)
assert CODE_BUNDLE.is_file(), f'Upload local code bundle to {CODE_BUNDLE}'
run('git', 'clone', CODE_BUNDLE, repo)
run('git', 'checkout', REPO_COMMIT, cwd=repo)
run('git', 'submodule', 'update', '--init', cwd=repo)
assert subprocess.check_output(['git','rev-parse','HEAD'], cwd=repo, text=True).strip().startswith(REPO_COMMIT)
assert subprocess.check_output(['git','rev-parse','HEAD'], cwd=repo/'victimagent', text=True).strip() == VICTIM_COMMIT
run('python', '-m', 'pip', 'install', '-q', '-e', '.[dev]', cwd=repo)


In [ ]:
required = ['crsp_dsf_spy_1993_2008.csv','crsp_dsf_spy_tlt_gld_2009_2025.csv','stkdistributions_spy_1993_2008.csv','stkdistributions_spy_tlt_gld_2009_2025.csv','DGS3MO_1993_2008.csv','DGS3MO_2009_2025.csv']
missing = [x for x in required if not (RAW/x).is_file()]
assert not missing, f'Missing Drive raw inputs: {missing}'
target = repo/'victimagent/data/raw'; target.mkdir(parents=True, exist_ok=True)
for name in required: shutil.copy2(RAW/name, target/name)
# Restore all 20 completed local records and their audit artifacts.
assert STATE_ARCHIVE.is_file(), f'Upload local state archive to {STATE_ARCHIVE}'
shutil.unpack_archive(STATE_ARCHIVE, repo)
# Also restore any newer state already synced by an earlier Colab session.
for rel in ['experiments/v9_ppo','artifacts/checkpoints/v9_ppo','artifacts/trajectories/v9_ppo']:
    src, dst = STATE/rel, repo/rel
    if src.exists(): shutil.copytree(src, dst, dirs_exist_ok=True)
run('python', 'scripts/run_ppo_v9.py', '--build-data', cwd=repo)
run('python', 'scripts/run_ppo_v9.py', '--preflight', cwd=repo)


In [ ]:
import concurrent.futures, json
arms = ['ppo_v9_gate_nav','ppo_v9_gate_relative']
folds = ['fold_1','fold_2','fold_3']
cells = [(a,f,s) for a in arms for f in folds for s in range(5)]
def paths(cell):
    a,f,s=cell; label=f'{a}_{f}_seed{s}_v9spec1'
    return repo/f'experiments/v9_ppo/records/{label}.json', repo/f'experiments/v9_ppo/attempts/{label}.json'
missing=[]
for cell in cells:
    record, attempt = paths(cell)
    if record.exists():
        assert json.loads(record.read_text())['status']=='PASS', f'Failed record: {record}'
    elif attempt.exists(): raise RuntimeError(f'Attempt without completed record; inspect, do not delete: {attempt}')
    else: missing.append(cell)
print('Completed:', 30-len(missing), 'Missing:', len(missing), missing)
MAX_PARALLEL = 2  # Colab CPU usually benefits from 1–2 workers, not four.
def train(cell):
    a,f,s=cell
    run('python','scripts/run_ppo_v9.py','--run','--arm',a,'--fold',f,'--seed',s,'--threads',1,cwd=repo)
    record,_=paths(cell); assert json.loads(record.read_text())['status']=='PASS'
    return cell
with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_PARALLEL) as pool:
    for cell in pool.map(train, missing):
        print('PASS', cell, flush=True)
        for rel in ['experiments/v9_ppo','artifacts/checkpoints/v9_ppo','artifacts/trajectories/v9_ppo']:
            shutil.copytree(repo/rel, STATE/rel, dirs_exist_ok=True)
run('python','scripts/run_ppo_v9.py','--aggregate',cwd=repo)
shutil.copytree(repo/'reports/v9_ppo', STATE/'reports/v9_ppo', dirs_exist_ok=True)
print((repo/'reports/v9_ppo/VALIDATION_SUMMARY.md').read_text())
